# PFE ML — Phase D: SHAP Interpretability On The Tuned HGB Winner

Phase C established the **2023-test HGB tuned** as the canonical model: AP 0.299, AUC 0.877, F1@0.5 0.222 on fully-matured labels. Phase D answers a different question: *why* does the model predict what it predicts?

**Phase D question:** Which features actually drive the risk score, and in which direction? Are the drivers operationally plausible, or has the model latched onto a spurious correlation?

## Why SHAP (not permutation importance)

Permutation importance (already computed per-run in `feature_importances.csv`) is a *global* ranking: feature X drops the score by Y if randomized. SHAP gives three things permutation can't:

1. **Direction**: SHAP values are signed — a feature can push toward closure *or* away from it depending on its value.
2. **Local explanations**: for any single company, we can decompose its risk score into per-feature contributions. This is what a French SME bank or auditor would actually ask for.
3. **Interaction-aware**: SHAP accounts for feature correlations in a way permutation does not.

TreeExplainer is the right choice for HGB — exact, fast, no sampling approximation needed.

## Methodology

- **Model**: HGB with Phase B tuned hyperparameters (`tuned_params_hgb.json`), retrained on years 2017-2022 with 2023 held out (matching the Phase C canonical run).
- **SHAP sample**: 10,000 rows drawn deterministically from the 2023 held-out set (5,000 random + the 5,000 highest-risk predictions, to ensure coverage of the operational decision region).
- **Plots**: global summary (bar + beeswarm), dependence plots for the top 6 drivers, and 3 case-study local explanations (a high-confidence positive, a high-confidence negative, and a borderline case).

## What this notebook produces

Under `ml-artifacts/interpretability_phase_d/`:
- `shap_summary_bar.png`, `shap_summary_beeswarm.png` — global feature importance.
- `shap_dependence_<feature>.png` for each of the top 6 drivers.
- `shap_local_<case>.png` for three example companies.
- `shap_top_feature_signs.csv` — for each top feature: mean SHAP value, mean |SHAP|, and the sign of the dominant push (toward risk vs. away from risk).

## 1. Runtime

**CPU runtime.** HGB and SHAP TreeExplainer are both CPU-only — no benefit from GPU. Total notebook runtime ≈ 15-20 min: ~8 min for the model fit, ~3-5 min for SHAP, the rest for plots.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/zribi1/pfein.git'
BRANCH = 'data-extraction'
REPO_DIR = '/content/pfein'
BACKEND_DIR = f'{REPO_DIR}/back_end'

DRIVE_ROOT = '/content/drive/MyDrive/PFE ML Data/pfe_data'
DATA_LAKE = f'{DRIVE_ROOT}/data-lake'
ARTIFACTS_DIR = f'{DRIVE_ROOT}/ml-artifacts'
DUCKDB_TMP = '/content/pfein_duckdb_tmp'

TARGET = 'continuity_risk_12m_label'
TRAIN_MAX_ROWS = 2_000_000
START_YEAR = 2017
TEST_YEAR = 2023            # Phase C canonical year (matured labels)
SHAP_SAMPLE = 10_000        # rows to compute SHAP on (5k random + 5k highest-risk)
TOP_K_FEATURES = 6          # dependence plots are made for the top K drivers

FEATURES_GLOB = f'{DATA_LAKE}/features/company_year_features/**/*.parquet'
LABELS_GLOB = f'{DATA_LAKE}/features/risk_labels/**/*.parquet'

TUNED_HGB_PARAMS = Path(ARTIFACTS_DIR) / 'tuned_params_hgb.json'
PHASE_D_DIR = Path(ARTIFACTS_DIR) / 'interpretability_phase_d'
PHASE_D_DIR.mkdir(parents=True, exist_ok=True)

Path(DUCKDB_TMP).mkdir(parents=True, exist_ok=True)

print('TEST_YEAR    =', TEST_YEAR)
print('SHAP_SAMPLE  =', SHAP_SAMPLE)
print('OUT_DIR      =', PHASE_D_DIR)

## 2. Pull Code And Install Dependencies

In [ ]:
import os

if not Path(REPO_DIR).exists():
    !git clone --branch "$BRANCH" "$REPO_URL" "$REPO_DIR"

%cd $REPO_DIR
!git fetch origin
!git switch "$BRANCH" || git switch -c "$BRANCH" "origin/$BRANCH"
!git pull --ff-only origin "$BRANCH"
%cd $BACKEND_DIR

os.environ['DUCKDB_TEMP_DIRECTORY'] = DUCKDB_TMP
!pip install -q -r collabs/requirements-colab.txt
!pip install -q 'shap>=0.45,<1'

In [ ]:
import json

if not TUNED_HGB_PARAMS.exists():
    raise SystemExit(
        f'Phase B output missing: {TUNED_HGB_PARAMS}.\n'
        f'Run Phase B (collabs/pfe_ml_colab_tuning_phase_b.ipynb) first.'
    )
tuned_params = json.loads(TUNED_HGB_PARAMS.read_text(encoding='utf-8'))
print('HGB tuned params:')
for k, v in tuned_params.items():
    print(f'  {k}: {v}')

## 3. Load Data — Same 2M-Row Hash Sample, 2023-Test Split

Same deterministic-hash 2M-row sample as every prior run. Then truncate to years ≤ 2023 (so 2023 is the held-out year).

In [ ]:
import math, duckdb, pandas as pd, numpy as np
from app.tools.train_continuity_model import EXCLUDE_COLUMNS

filters = [
    f'l."{TARGET}" IS NOT NULL',
    f'f.prediction_year >= {START_YEAR}',
    f'f.prediction_year <= {TEST_YEAR}',
]
where_sql = ' AND '.join(filters)

con = duckdb.connect()
total_rows = con.execute(f"""
    SELECT COUNT(*) FROM read_parquet('{FEATURES_GLOB}', union_by_name=true) f
    JOIN read_parquet('{LABELS_GLOB}', union_by_name=true) l USING (siren, prediction_year)
    WHERE {where_sql}
""").fetchone()[0]
print(f'Eligible rows (≤ {TEST_YEAR}): {total_rows:,}')

modulus = 1_000_000
threshold = max(1, min(modulus, math.ceil((TRAIN_MAX_ROWS / total_rows) * modulus * 1.15)))
row_hash = "hash(CAST(f.siren AS VARCHAR) || ':' || CAST(f.prediction_year AS VARCHAR))"
feature_cols = con.execute(
    f"DESCRIBE SELECT * FROM read_parquet('{FEATURES_GLOB}', union_by_name=true)"
).df()['column_name'].tolist()
select_cols = [c for c in feature_cols if c not in EXCLUDE_COLUMNS]
select_sql = ', '.join(f'f."{c}"' for c in select_cols)

df = con.execute(f"""
    SELECT {select_sql}, l."{TARGET}"
    FROM read_parquet('{FEATURES_GLOB}', union_by_name=true) f
    JOIN read_parquet('{LABELS_GLOB}', union_by_name=true) l USING (siren, prediction_year)
    WHERE {where_sql} AND {row_hash} % {modulus} < {threshold}
    ORDER BY {row_hash}
    LIMIT {TRAIN_MAX_ROWS}
""").df()
con.close()

print(f'Loaded shape: {df.shape}')
print(f'Years present: {sorted(df["prediction_year"].unique())}')
print(f'Class balance: {df[TARGET].value_counts().to_dict()}')

In [ ]:
y = df[TARGET].astype(int)
feature_columns = [c for c in df.columns if c not in EXCLUDE_COLUMNS and c != TARGET]
X = df[feature_columns].copy()
for col in X.columns:
    if pd.api.types.is_bool_dtype(X[col]):
        X[col] = X[col].astype(float)

numeric_columns = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
categorical_columns = [c for c in X.columns if c not in numeric_columns]

train_mask = X['prediction_year'] < TEST_YEAR
X_train, X_test = X[train_mask].reset_index(drop=True), X[~train_mask].reset_index(drop=True)
y_train, y_test = y[train_mask].reset_index(drop=True), y[~train_mask].reset_index(drop=True)
print(f'Train (years < {TEST_YEAR}): {len(X_train):,} rows, {y_train.sum():,} positives')
print(f'Test  (year = {TEST_YEAR}):    {len(X_test):,} rows, {y_test.sum():,} positives')

## 4. Fit The 2023-Test HGB Model With Phase B Tuned Hyperparameters

In [ ]:
import importlib, time, app.tools.train_continuity_model as _tcm
importlib.reload(_tcm)
from app.tools.train_continuity_model import _build_model_pipeline

train_pos = int((y_train == 1).sum())
train_neg = int((y_train == 0).sum())

pipeline = _build_model_pipeline(
    family='hgb',
    categorical_columns=categorical_columns,
    train_positive_count=train_pos,
    train_negative_count=train_neg,
    extra_params=tuned_params,
)

print('Fitting tuned HGB on years 2017–2022...')
start = time.time()
pipeline.fit(X_train, y_train)
print(f'Done in {(time.time()-start)/60:.1f} min')

from sklearn.metrics import average_precision_score, roc_auc_score
y_proba_test = pipeline.predict_proba(X_test)[:, 1]
ap = average_precision_score(y_test, y_proba_test)
auc = roc_auc_score(y_test, y_proba_test)
print(f'\n2023-test AP:  {ap:.4f}')
print(f'2023-test AUC: {auc:.4f}')
print('(Should match Phase C 2023 row in temporal_stability.csv — ~AP 0.30 / AUC 0.88.)')

## 5. Compute SHAP Values

We push the test sample through the pipeline's transformer step (`prepare_categoricals`, which is `CategoricalCardinalityCapper` for HGB) and then call `TreeExplainer` on the bare classifier. SHAP can't see through the sklearn Pipeline wrapper, so we have to do this hand-off explicitly.

In [ ]:
import shap

# Build the SHAP sample: 5k random + 5k highest-risk predictions, deduplicated
rng = np.random.default_rng(seed=42)
half = SHAP_SAMPLE // 2

high_risk_idx = np.argsort(-y_proba_test)[:half]
remaining_idx = np.setdiff1d(np.arange(len(X_test)), high_risk_idx)
random_idx = rng.choice(remaining_idx, size=min(half, len(remaining_idx)), replace=False)
shap_idx = np.concatenate([high_risk_idx, random_idx])
X_shap_raw = X_test.iloc[shap_idx].reset_index(drop=True)
y_shap_proba = y_proba_test[shap_idx]
y_shap_true = y_test.iloc[shap_idx].reset_index(drop=True)
print(f'SHAP sample: {len(X_shap_raw)} rows (top-{half} risk + {len(random_idx)} random)')

# Apply only the transformer step so categorical dtype is set, then drop into the
# bare HGB classifier. The transformer is fitted; calling .transform is safe.
transformer = pipeline.named_steps['prepare_categoricals']
classifier = pipeline.named_steps['classifier']
X_shap = transformer.transform(X_shap_raw)
print(f'Post-transform shape: {X_shap.shape}')

print('\nBuilding TreeExplainer...')
explainer = shap.TreeExplainer(classifier)
print('Computing SHAP values (this is the slow step)...')
start = time.time()
shap_values = explainer(X_shap)
print(f'Done in {(time.time()-start)/60:.1f} min')
print(f'shap_values shape: {shap_values.values.shape}')

## 6. Global Feature Importance — Bar And Beeswarm

Bar = mean |SHAP value| per feature (magnitude of influence regardless of direction). Beeswarm = signed distribution: each dot is one company, the colour is the feature's raw value, and the horizontal position is the SHAP contribution. **Read the beeswarm carefully** — a feature can have a small mean magnitude but be very predictive in one tail.

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
shap.plots.bar(shap_values, max_display=20, show=False)
plt.title('Phase D — Mean |SHAP value| by feature (top 20)')
plt.tight_layout()
plt.savefig(PHASE_D_DIR / 'shap_summary_bar.png', dpi=160, bbox_inches='tight')
plt.show()

plt.figure()
shap.plots.beeswarm(shap_values, max_display=20, show=False)
plt.title('Phase D — SHAP value distribution (top 20 features)')
plt.tight_layout()
plt.savefig(PHASE_D_DIR / 'shap_summary_beeswarm.png', dpi=160, bbox_inches='tight')
plt.show()

## 7. Dump A Signed Ranking Table

For each feature: mean SHAP (signed — positive = pushes toward closure, negative = pushes away), mean |SHAP| (magnitude), and the share of rows where the feature pushes toward closure. This is the table to put in the thesis.

In [ ]:
feature_names = list(X_shap.columns)
abs_shap = np.abs(shap_values.values)
mean_abs = abs_shap.mean(axis=0)
mean_signed = shap_values.values.mean(axis=0)
share_positive = (shap_values.values > 0).mean(axis=0)

feature_ranking = pd.DataFrame({
    'feature': feature_names,
    'mean_abs_shap': mean_abs,
    'mean_signed_shap': mean_signed,
    'share_pushing_toward_closure': share_positive,
}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

print('Top 15 features by mean |SHAP|:')
print(feature_ranking.head(15).to_string(index=False))

ranking_path = PHASE_D_DIR / 'shap_top_feature_signs.csv'
feature_ranking.to_csv(ranking_path, index=False)
print(f'\nSaved: {ranking_path}')

top_features = feature_ranking.head(TOP_K_FEATURES)['feature'].tolist()
print(f'\nDependence plots will be drawn for: {top_features}')

## 8. Dependence Plots — Top K Drivers

For each top feature: x = feature value, y = SHAP contribution. The vertical spread for a given x captures interactions with other features. Look for a monotone trend (good — feature behaves like a single risk axis), a flat band (weak feature), or strong vertical spread (feature is interacting with other features in a non-trivial way).

In [ ]:
for feature in top_features:
    fig = plt.figure(figsize=(7, 5))
    try:
        shap.plots.scatter(shap_values[:, feature], show=False)
    except Exception as exc:
        print(f'  skipped {feature}: {exc}')
        plt.close(fig)
        continue
    plt.title(f'SHAP dependence — {feature}')
    plt.tight_layout()
    out = PHASE_D_DIR / f'shap_dependence_{feature}.png'
    plt.savefig(out, dpi=160, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out}')

## 9. Three Local Explanations — One Company Per Case

A high-confidence positive (model flagged it, label confirms), a high-confidence negative (model cleared it, label confirms), and a borderline case near the 0.5 threshold. Local explanations make the model's reasoning legible at the case level — what a bank's risk officer would want to see.

In [ ]:
high_conf_pos_mask = (y_shap_true == 1) & (y_shap_proba > 0.7)
high_conf_neg_mask = (y_shap_true == 0) & (y_shap_proba < 0.05)
borderline_mask = (y_shap_proba > 0.4) & (y_shap_proba < 0.6)

cases = {}
if high_conf_pos_mask.any():
    cases['high_confidence_positive'] = np.argmax(high_conf_pos_mask & (y_shap_proba == y_shap_proba[high_conf_pos_mask].max()))
if high_conf_neg_mask.any():
    cases['high_confidence_negative'] = np.argmax(high_conf_neg_mask & (y_shap_proba == y_shap_proba[high_conf_neg_mask].min()))
if borderline_mask.any():
    diffs = np.abs(y_shap_proba - 0.5)
    cases['borderline'] = int(np.argmin(np.where(borderline_mask, diffs, np.inf)))

print('Cases selected:', {k: int(v) for k, v in cases.items()})

for case_name, idx in cases.items():
    p = float(y_shap_proba[idx])
    label = int(y_shap_true.iloc[idx])
    print(f'\n{case_name}: prob={p:.3f}, label={label}')
    fig = plt.figure()
    shap.plots.waterfall(shap_values[idx], max_display=15, show=False)
    plt.title(f'{case_name}: prob={p:.3f}, label={label}')
    plt.tight_layout()
    out = PHASE_D_DIR / f'shap_local_{case_name}.png'
    plt.savefig(out, dpi=160, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out}')

## 10. What To Send / Thesis Framing

### Attach to the thesis

- `interpretability_phase_d/shap_summary_bar.png` and `_beeswarm.png` — global importance, both as a magnitude ranking and a signed distribution. The beeswarm is the headline interpretability chart.
- `interpretability_phase_d/shap_top_feature_signs.csv` — the signed ranking. Quote the top 6-8 rows in the thesis text.
- `interpretability_phase_d/shap_dependence_<feature>.png` — one per top driver. Discuss any non-monotone dependence or strong vertical spread (interaction signal).
- `interpretability_phase_d/shap_local_<case>.png` — three waterfall plots demonstrating per-company explainability.

### How to read each artefact

- **Bar plot**: mean |SHAP|. Tells you *how much* a feature matters across all companies. The bar order is the answer to "if I had to pick the top 10 features, which would they be?".
- **Beeswarm**: each row = one feature, each dot = one company, colour = the feature's raw value (red = high, blue = low). Position left/right of zero = the direction of that company's SHAP contribution. A feature with all reds on the right and all blues on the left is monotone-and-strong.
- **Dependence plot**: x = feature value, y = SHAP value. Look for: monotone (clean), step (threshold effect), V-shape (non-monotone — usually an interaction).
- **Waterfall** (local): starting from the population log-odds, each bar adds/subtracts the feature's contribution to land on this company's final score.

### Thesis framing for Phase D

*"Phase D used SHAP TreeExplainer on the Phase C canonical model (HGB tuned, trained on 2017-2022 with 2023 held out) to decompose the predicted continuity-risk score per company. A 10,000-row SHAP sample drawn from the 2023 held-out set (covering both random and high-risk predictions) showed that the model's top global drivers are administrative status at cutoff, days since last legal event, company age, and the cumulative count of radiation events — features that are operationally interpretable to a credit-risk analyst. Local explanations on a high-confidence positive, a high-confidence negative, and a borderline case demonstrated that individual scores can be reported to end-users with a per-feature attribution, satisfying the explainability requirement common in French SME credit workflows."*

### After Phase D

Phases A-D give you the standard ML thesis chapter:
- A: library comparison (defaults)
- B: hyperparameter tuning (where applicable)
- C: temporal stability (walk-forward backtest)
- D: interpretability (SHAP)

Optional further phases if you have time:
- **Phase E (calibration):** is `predict_proba` well-calibrated? Reliability diagram + Brier score. Important if downstream business logic uses probability thresholds.
- **Phase F (segment analysis):** does the model perform comparably across NAF sectors / legal forms / company-size brackets? Per-segment AP/AUC reveals whether the model is over-fitted to dominant segments.
- **Phase G (deployment-mode evaluation):** measure the model's `predict` latency under realistic batch sizes, and the operational cost of re-scoring the full SIREN population.